<a href="https://colab.research.google.com/github/vibrito/qualidade-de-software/blob/main/fifa_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#  Classificação de Jogadores FIFA

## Objetivo
Classificar jogadores em:
- Zegueiro
- Meio-campista
- Atacante

Utilizando Machine Learning com Scikit-Learn.


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report


##  Carregamento dos dados

In [ ]:

URL = "https://raw.githubusercontent.com/KeithGalli/matplotlib_tutorial/master/fifa_data.csv"
df = pd.read_csv(URL)

df.head()


,Unnamed: 0,ID,Name,Age,Photo,Nationality,Flag,Overall,Potential,Club,...,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause
0,0,158023,L. Messi,31,https://cdn.sofifa.org/players/4/19/158023.png,Argentina,https://cdn.sofifa.org/flags/52.png,94,94,FC Barcelona,...,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0,€226.5M
1,1,20801,Cristiano Ronaldo,33,https://cdn.sofifa.org/players/4/19/20801.png,Portugal,https://cdn.sofifa.org/flags/38.png,94,94,Juventus,...,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0,€127.1M
2,2,190871,Neymar Jr,26,https://cdn.sofifa.org/players/4/19/190871.png,Brazil,https://cdn.sofifa.org/flags/54.png,92,93,Paris Saint-Germain,...,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0,€228.1M
3,3,193080,De Gea,27,https://cdn.sofifa.org/players/4/19/193080.png,Spain,https://cdn.sofifa.org/flags/45.png,91,93,Manchester United,...,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0,€138.6M
4,4,192985,K. De Bruyne,27,https://cdn.sofifa.org/players/4/19/192985.png,Belgium,https://cdn.sofifa.org/flags/7.png,91,92,Manchester City,...,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0,€196.4M


##  Pré-processamento

In [ ]:

df.columns = df.columns.str.replace(" ", "").str.strip()

def categorize_position(pos):
    if pd.isna(pos):
        return np.nan

    pos = str(pos).upper()

    if "GK" in pos:
        return np.nan
    if any(p in pos for p in ["CB", "LB", "RB", "LWB", "RWB"]):
        return "Zagueiro"
    if any(p in pos for p in ["CM", "CDM", "CAM", "LM", "RM", "LW", "RW"]):
        return "Meio-campista"
    return "Atacante"

df["Position_Group"] = df["Position"].apply(categorize_position)
df = df.dropna(subset=["Position_Group"])


## Features utilizadas

In [ ]:

features = [
    "SprintSpeed",
    "Finishing",
    "ShortPassing",
    "Vision",
    "Marking",
    "StandingTackle"
]

X = df[features]
y = df["Position_Group"]


## Divisão treino/teste

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Pipeline

In [ ]:

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), features)
])


## Modelos

In [ ]:

models = {
    "KNN": Pipeline([
        ("pre", preprocessor),
        ("model", KNeighborsClassifier())
    ]),
    "DecisionTree": Pipeline([
        ("pre", preprocessor),
        ("model", DecisionTreeClassifier())
    ]),
    "NaiveBayes": Pipeline([
        ("pre", preprocessor),
        ("model", GaussianNB())
    ]),
    "SVM": Pipeline([
        ("pre", preprocessor),
        ("model", SVC())
    ])
}


## Treinamento e avaliação

In [ ]:

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(f"\n{name}")
    print(f"Acurácia: {acc:.4f}")
    print(classification_report(y_test, preds))



KNN
Acurácia: 0.8031
               precision    recall  f1-score   support

     Atacante       0.75      0.64      0.69       640
Meio-campista       0.77      0.82      0.79      1412
     Zagueiro       0.87      0.88      0.87      1173

     accuracy                           0.80      3225
    macro avg       0.80      0.78      0.79      3225
 weighted avg       0.80      0.80      0.80      3225


DecisionTree
Acurácia: 0.7318
               precision    recall  f1-score   support

     Atacante       0.59      0.61      0.60       640
Meio-campista       0.72      0.72      0.72      1412
     Zagueiro       0.83      0.81      0.82      1173

     accuracy                           0.73      3225
    macro avg       0.71      0.71      0.71      3225
 weighted avg       0.73      0.73      0.73      3225


NaiveBayes
Acurácia: 0.7507
               precision    recall  f1-score   support

     Atacante       0.63      0.72      0.67       640
Meio-campista       0.78      0

## Melhor modelo

In [ ]:

best_model_name = max(results, key=results.get)
print("Melhor modelo:", best_model_name)


Melhor modelo: SVM



## Segurança

- Evitamos o uso de dados sensiveis. Os nomes dos jogadores, times e nacionalidades  não aparecem no front-end. São usados apenas números.
- Não guardamos os dados dos usuários e nem os dados da consulta.
- Validamos os dados e temos tratamento de erro no front-end.
